# Aula 8: avaliação prática de pandas, versão professor

As mesmas 20 perguntas da versão do aluno, com a resposta já resolvida em
cada célula de código, e o valor esperado registrado logo abaixo do
enunciado. Use para conferir as respostas que os alunos colarem no
formulário.

**Não compartilhe este notebook com a turma.**

Todas as 20 respostas foram calculadas e conferidas contra os
1.000.090 dados reais deste pacote antes de este notebook ser
considerado pronto.

## Preparação

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
PASTA_DADOS = "/content/drive/MyDrive/AI Talent Academy/Aula 08/dados/csv"

In [ ]:
import pandas as pd

categorias = pd.read_csv(f"{PASTA_DADOS}/categorias.csv")
clientes = pd.read_csv(f"{PASTA_DADOS}/clientes.csv")
lojistas = pd.read_csv(f"{PASTA_DADOS}/lojistas.csv")
cartoes = pd.read_csv(f"{PASTA_DADOS}/cartoes.csv")
transacoes = pd.read_csv(f"{PASTA_DADOS}/transacoes.csv")
transacoes["data_hora"] = pd.to_datetime(transacoes["data_hora"])

print("transacoes:", transacoes.shape)

## Carregando e explorando

### Pergunta 1

Quantas transações existem na tabela `transacoes`?

**Resposta esperada:** 1.000.090

In [ ]:
len(transacoes)

### Pergunta 2

Qual o tipo de dado (`dtype`) da coluna `valor` depois de carregada? E da coluna `data_hora`, antes de qualquer conversão de tipo?

**Resposta esperada:** valor: float64. data_hora antes de converter: object (texto)

In [ ]:
print(transacoes["valor"].dtype)
print(pd.read_csv(f"{PASTA_DADOS}/transacoes.csv")["data_hora"].dtype)

### Pergunta 3

Qual o valor médio das transações? E o desvio padrão?

**Resposta esperada:** média ≈ R$121,72. desvio padrão ≈ R$159,20

In [ ]:
print(transacoes["valor"].mean())
print(transacoes["valor"].std())

### Pergunta 4

Quantos valores ausentes (NaN) existem na coluna `telefone` da tabela `clientes`?

**Resposta esperada:** 251

In [ ]:
clientes["telefone"].isna().sum()

## Filtrando e selecionando

### Pergunta 5

Quantas transações têm valor acima de R$1.000?

**Resposta esperada:** 4.645

In [ ]:
(transacoes["valor"] > 1000).sum()

### Pergunta 6

Quantos canais distintos existem na tabela `transacoes`, e quantas transações tem cada um?

**Resposta esperada:** 2 canais. presencial: 650.433, online: 349.657

In [ ]:
transacoes["canal"].value_counts()

### Pergunta 7

Qual o nome do cliente com `id_cliente` igual a 1?

**Resposta esperada:** Brenda Alves

In [ ]:
clientes.set_index("id_cliente").loc[1, "nome"]

### Pergunta 8

Quantos lojistas distintos aparecem na tabela `transacoes`? (repare que pode ser menor que o total de lojistas cadastrados)

**Resposta esperada:** 294 (de um total de 300 lojistas cadastrados)

In [ ]:
transacoes["id_lojista"].nunique()

## Agregando

### Pergunta 9

Qual `id_lojista` tem o maior valor total de transações somado?

**Resposta esperada:** id_lojista 168 (Express Supermercado)

In [ ]:
resumo_lojista = transacoes.groupby("id_lojista")["valor"].sum()
resumo_lojista.idxmax()

### Pergunta 10

Para o lojista líder da pergunta anterior, quantas transações ele tem, e qual o valor total?

**Resposta esperada:** 3.571 transações, R$450.205,41 no total

In [ ]:
transacoes[transacoes["id_lojista"] == resumo_lojista.idxmax()]["valor"].agg(["count", "sum"])

### Pergunta 11

Juntando `transacoes` com `lojistas` e `categorias`, qual categoria de lojista tem o maior valor total faturado?

**Resposta esperada:** Vestuário, aproximadamente R$14.519.745

In [ ]:
completo_categoria = (transacoes
    .merge(lojistas[["id_lojista", "id_categoria"]], on="id_lojista")
    .merge(categorias, on="id_categoria"))
completo_categoria.groupby("nome")["valor"].sum().idxmax()

### Pergunta 12

Usando `pivot_table`, qual o valor total movimentado em cada canal (presencial e online)?

**Resposta esperada:** presencial ≈ R$79.143.136, online ≈ R$42.585.933

In [ ]:
pd.pivot_table(transacoes, index="canal", values="valor", aggfunc="sum")

## Juntando tabelas

### Pergunta 13

Juntando `transacoes`, `cartoes` e `clientes`, qual o nome do cliente que mais gastou no total, somando todas as transações dele?

**Resposta esperada:** Lavínia Novaes, aproximadamente R$271.674,81

In [ ]:
completo_cliente = (transacoes
    .merge(cartoes[["id_cartao", "id_cliente"]], on="id_cartao")
    .merge(clientes[["id_cliente", "nome"]], on="id_cliente"))
completo_cliente.groupby("nome")["valor"].sum().sort_values(ascending=False).head(1)

### Pergunta 14

Quantos clientes nunca fizeram nenhuma transação em nenhum cartão?

**Resposta esperada:** 15

In [ ]:
cartoes_com_transacao = set(transacoes["id_cartao"].unique())
clientes_ativos = cartoes[cartoes["id_cartao"].isin(cartoes_com_transacao)]["id_cliente"].unique()
len(set(clientes["id_cliente"]) - set(clientes_ativos))

### Pergunta 15

Quantos lojistas nunca receberam nenhuma transação?

**Resposta esperada:** 6

In [ ]:
lojistas_com_transacao = set(transacoes["id_lojista"].unique())
len(set(lojistas["id_lojista"]) - lojistas_com_transacao)

## Visualizando

### Pergunta 16

Qual o valor no percentil 95 das transações? Ou seja, abaixo de qual valor ficam 95% das transações da base?

**Resposta esperada:** aproximadamente R$382,70

In [ ]:
transacoes["valor"].quantile(0.95)

### Pergunta 17

Plote um histograma do valor das transações (com Matplotlib ou Seaborn). Olhando o gráfico, a distribuição é simétrica ou puxada para a direita (cauda longa de valores altos)?

**Resposta esperada:** Puxada para a direita: a maioria das transações é de valor baixo, com uma cauda longa de transações raras e caras

In [ ]:
import seaborn as sns
sns.histplot(transacoes["valor"])

## Investigação

### Pergunta 18

Calcule a taxa de contestação (proporção de transações com status "contestada") de cada lojista. Qual lojista tem a maior taxa, e qual é essa taxa, aproximadamente?

**Resposta esperada:** id_lojista 141 (Express Posto), aproximadamente 23,7%

In [ ]:
taxa_contestacao = transacoes.groupby("id_lojista")["status"].apply(
    lambda s: (s == "contestada").mean())
taxa_contestacao.sort_values(ascending=False).head(1)

### Pergunta 19

Para o lojista mais suspeito da pergunta anterior, qual o percentual das transações dele que acontece entre 1h e 3h da manhã?

**Resposta esperada:** aproximadamente 69%

In [ ]:
lojista_suspeito = taxa_contestacao.idxmax()
horas = transacoes.loc[transacoes["id_lojista"] == lojista_suspeito, "data_hora"].dt.hour
horas.between(1, 3).mean()

### Pergunta 20

Procure por "teste de cartão": cartões com 4 ou mais transações de valor até R$5, no mesmo lojista, no mesmo dia. Quantos cartões distintos mostram esse padrão?

**Resposta esperada:** 15

In [ ]:
pequenas = transacoes[transacoes["valor"] <= 5]
rajadas = (pequenas
    .assign(dia=pequenas["data_hora"].dt.date)
    .groupby(["id_cartao", "id_lojista", "dia"])
    .size()
    .reset_index(name="qtd"))
rajadas[rajadas["qtd"] >= 4]["id_cartao"].nunique()

## Fim do gabarito

Todas as 20 respostas foram conferidas contra os dados reais deste
pacote (1.000.090 transações, com a fraude plantada descrita em
`historia_anomalia.md`). Se os dados forem regerados com um `--seed`
diferente, rode este notebook de novo antes de usar como gabarito,
porque os valores esperados podem mudar.